# Notebook 08: MLflow Regression — RUL Estimation
## Purpose: Train + compare 3 models to predict Remaining Useful Life (RUL)
## Input:  workspace.predictive_maintenance.gold_ml_input
## Output: Best model registered in MLflow Model Registry

## ML Task: Regression
## Target:  RUL (continuous — cycles remaining before failure)
## Models:  Linear Regression vs Random Forest vs XGBoost
## Metrics: RMSE, MAE, R2

## Expert Upgrade: Quantile Regression
## Instead of one RUL prediction, we give a confidence interval:
## "Machine 47 will fail between 23 and 41 cycles — 80% confidence"
## This is how production ML systems handle uncertainty

In [0]:
%pip install xgboost scikit-learn mlflow

dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from mlflow.models.signature import infer_signature
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Load ML input table
df = spark.table("workspace.predictive_maintenance.gold_ml_input")
print(f"Loaded: {df.count():,} rows | {len(df.columns)} columns")

In [0]:
EXCLUDE_COLS = [
    "unit_id", "cycle", "RUL", "fail_30", "fail_15",
    "source_dataset", "setting_1", "setting_2", "setting_3"
]

FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]
TARGET_COL   = "RUL"

print(f"Feature columns: {len(FEATURE_COLS)}")
print(f"Target column:   {TARGET_COL} (continuous)")

pdf = df.select(FEATURE_COLS + [TARGET_COL]).toPandas()
pdf = pdf.fillna(0)

print(f"\nRUL DISTRIBUTION")
print(f"Min:  {pdf[TARGET_COL].min()}")
print(f"Max:  {pdf[TARGET_COL].max()}")
print(f"Mean: {pdf[TARGET_COL].mean():.1f}")
print(f"Std:  {pdf[TARGET_COL].std():.1f}")

In [0]:
X = pdf[FEATURE_COLS]
y = pdf[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train set: {X_train.shape[0]:,} rows")
print(f"Test set:  {X_test.shape[0]:,} rows")
print(f"Features:  {X_train.shape[1]}")

In [0]:
def evaluate_regression(model, X_test, y_test,
                        model_name, use_scaled=False):
    X      = X_test_scaled if use_scaled else X_test
    y_pred = model.predict(X)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    print(f"\n{model_name} RESULTS")
    print(f"RMSE: {rmse:.2f} cycles")
    print(f"MAE:  {mae:.2f} cycles")
    print(f"R2:   {r2:.4f}")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(y_test, y_pred, alpha=0.3, color='#1565C0', s=10)
    ax.plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()],
            'r--', linewidth=2, label='Perfect prediction')
    ax.set_xlabel("Actual RUL")
    ax.set_ylabel("Predicted RUL")
    ax.set_title(f"{model_name} — Actual vs Predicted RUL",
                 fontweight='bold')
    ax.legend()
    plt.tight_layout()
    path = f"/tmp/{model_name.replace(' ', '_')}_rul.png"
    plt.savefig(path)
    plt.show()
    plt.close()

    return {"rmse": rmse, "mae": mae, "r2": r2,
            "y_pred": y_pred, "plot_path": path}

print("Helper function ready")

In [0]:
mlflow.set_experiment("/Shared/PredMaint-RUL")
print("MLflow Experiment set: PredMaint-RUL")

In [0]:
with mlflow.start_run(run_name="Linear_Regression") as run:

    lr = LinearRegression()
    lr.fit(X_train_scaled, y_train)

    metrics = evaluate_regression(
        lr, X_test, y_test, "Linear Regression", use_scaled=True
    )

    mlflow.log_param("model_type",     "LinearRegression")
    mlflow.log_param("features_count", len(FEATURE_COLS))
    mlflow.log_metric("rmse", metrics["rmse"])
    mlflow.log_metric("mae",  metrics["mae"])
    mlflow.log_metric("r2",   metrics["r2"])
    mlflow.log_artifact(metrics["plot_path"])

    signature = infer_signature(
        X_train_scaled, lr.predict(X_train_scaled)
    )
    mlflow.sklearn.log_model(
        lr, "linear_regression_model", signature=signature
    )

    lr_run_id = run.info.run_id
    print(f"\nRun 1 logged — Linear Regression")
    print(f"RMSE={metrics['rmse']:.2f} | R2={metrics['r2']:.4f}")

In [0]:
with mlflow.start_run(run_name="Random_Forest_Regressor") as run:

    rf = RandomForestRegressor(
        n_estimators=100, max_depth=10,
        random_state=42, n_jobs=-1
    )
    rf.fit(X_train, y_train)

    metrics = evaluate_regression(
        rf, X_test, y_test, "Random Forest"
    )

    mlflow.log_param("model_type",     "RandomForestRegressor")
    mlflow.log_param("n_estimators",   100)
    mlflow.log_param("max_depth",      10)
    mlflow.log_param("features_count", len(FEATURE_COLS))
    mlflow.log_metric("rmse", metrics["rmse"])
    mlflow.log_metric("mae",  metrics["mae"])
    mlflow.log_metric("r2",   metrics["r2"])
    mlflow.log_artifact(metrics["plot_path"])

    signature = infer_signature(
        X_train, rf.predict(X_train)
    )
    mlflow.sklearn.log_model(
        rf, "random_forest_model",
        signature=signature,
        input_example=X_train.iloc[:5]
    )

    rf_run_id = run.info.run_id
    print(f"\nRun 2 logged — Random Forest")
    print(f"RMSE={metrics['rmse']:.2f} | R2={metrics['r2']:.4f}")

In [0]:
with mlflow.start_run(run_name="XGBoost_Regressor") as run:

    xgb_reg = xgb.XGBRegressor(
        n_estimators=200, max_depth=6,
        learning_rate=0.1, subsample=0.8,
        colsample_bytree=0.8, random_state=42
    )
    xgb_reg.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    metrics = evaluate_regression(
        xgb_reg, X_test, y_test, "XGBoost"
    )

    mlflow.log_param("model_type",       "XGBoostRegressor")
    mlflow.log_param("n_estimators",     200)
    mlflow.log_param("max_depth",        6)
    mlflow.log_param("learning_rate",    0.1)
    mlflow.log_param("features_count",   len(FEATURE_COLS))
    mlflow.log_metric("rmse", metrics["rmse"])
    mlflow.log_metric("mae",  metrics["mae"])
    mlflow.log_metric("r2",   metrics["r2"])
    mlflow.log_artifact(metrics["plot_path"])

    signature = infer_signature(
        X_train, xgb_reg.predict(X_train)
    )
    mlflow.xgboost.log_model(
        xgb_reg, "xgboost_regressor_model",
        signature=signature,
        input_example=X_train.iloc[:5]
    )

    xgb_rmse   = metrics["rmse"]
    xgb_r2     = metrics["r2"]
    xgb_run_id = run.info.run_id

    print(f"\nRun 3 logged — XGBoost")
    print(f"RMSE={metrics['rmse']:.2f} | R2={metrics['r2']:.4f}")

In [0]:
print("UNCERTAINTY QUANTIFICATION — Quantile Regression")
print("Training 3 quantile models...\n")

with mlflow.start_run(run_name="XGBoost_Quantile") as quant_run:

    xgb_q10 = xgb.XGBRegressor(
        objective='reg:quantileerror', quantile_alpha=0.1,
        n_estimators=200, max_depth=6,
        learning_rate=0.1, random_state=42
    )
    xgb_q10.fit(X_train, y_train, verbose=False)

    xgb_q50 = xgb.XGBRegressor(
        objective='reg:quantileerror', quantile_alpha=0.5,
        n_estimators=200, max_depth=6,
        learning_rate=0.1, random_state=42
    )
    xgb_q50.fit(X_train, y_train, verbose=False)

    xgb_q90 = xgb.XGBRegressor(
        objective='reg:quantileerror', quantile_alpha=0.9,
        n_estimators=200, max_depth=6,
        learning_rate=0.1, random_state=42
    )
    xgb_q90.fit(X_train, y_train, verbose=False)

    rul_lower  = xgb_q10.predict(X_test)
    rul_median = xgb_q50.predict(X_test)
    rul_upper  = xgb_q90.predict(X_test)

    interval_width = np.mean(rul_upper - rul_lower)
    median_rmse    = np.sqrt(mean_squared_error(y_test, rul_median))

    mlflow.log_param("model_type",       "XGBoost_Quantile")
    mlflow.log_param("quantiles",        "[0.1, 0.5, 0.9]")
    mlflow.log_param("confidence_level", "80%")
    mlflow.log_metric("interval_width",  interval_width)
    mlflow.log_metric("median_rmse",     median_rmse)

    quant_run_id = quant_run.info.run_id

    print(f"Quantile models trained")
    print(f"Average interval width: {interval_width:.1f} cycles")
    print(f"Median RMSE:            {median_rmse:.2f} cycles")

In [0]:
results_df = pd.DataFrame({
    'actual_rul':     y_test.values,
    'rul_lower':      np.maximum(0, rul_lower),
    'rul_median':     np.maximum(0, rul_median),
    'rul_upper':      np.maximum(0, rul_upper),
    'interval_width': rul_upper - rul_lower
})

near_failure = results_df[results_df['rul_median'] < 50].head(10)

print("MACHINES NEAR FAILURE — UNCERTAINTY PREDICTIONS")
print(f"{'Actual':>8} {'Lower':>8} {'Median':>8} {'Upper':>8} {'Width':>8}")
print("-" * 50)

for _, row in near_failure.iterrows():
    print(f"{row['actual_rul']:>8.0f} "
          f"{row['rul_lower']:>8.0f} "
          f"{row['rul_median']:>8.0f} "
          f"{row['rul_upper']:>8.0f} "
          f"{row['interval_width']:>8.0f}")

sample = near_failure.iloc[0]
print(f"\nExample interpretation:")
print(f"'This machine will fail between "
      f"{max(0, sample['rul_lower']):.0f} and "
      f"{sample['rul_upper']:.0f} cycles")
print(f" with 80% confidence.")
print(f" Schedule maintenance in "
      f"{max(0, sample['rul_lower']):.0f} cycles.'")

In [0]:
fig, ax = plt.subplots(figsize=(12, 6))

idx     = np.argsort(y_test.values)[:200]
x_range = np.arange(len(idx))

ax.fill_between(
    x_range, rul_lower[idx], rul_upper[idx],
    alpha=0.3, color='#1565C0',
    label='80% Confidence Interval'
)
ax.plot(x_range, rul_median[idx],
        color='#1565C0', linewidth=2,
        label='Predicted RUL (median)')
ax.plot(x_range, y_test.values[idx],
        color='red', linewidth=1.5,
        linestyle='--', label='Actual RUL')

ax.set_title(
    "RUL Prediction with Uncertainty Intervals\n"
    "Blue band = 80% confidence interval",
    fontweight='bold', fontsize=13
)
ax.set_xlabel("Machine samples (sorted by RUL)")
ax.set_ylabel("Remaining Useful Life (cycles)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

path = "/tmp/quantile_uncertainty.png"
plt.savefig(path, bbox_inches='tight')
plt.show()

with mlflow.start_run(run_id=quant_run_id):
    mlflow.log_artifact(path)

print("Uncertainty plot saved to MLflow artifacts")
print("VIDEO DEMO MOMENT — show this chart + say the interpretation")

In [0]:
print("=" * 55)
print("MODEL COMPARISON — PredMaint-RUL")
print("=" * 55)
print(f"{'Model':<28} {'RMSE':>8} {'R2':>8}")
print("-" * 55)

models_reg = {
    "Linear Regression": (lr,      True),
    "Random Forest":     (rf,      False),
    "XGBoost":          (xgb_reg, False),
}

best_rmse = float('inf')
best_name = ""

for name, (model, scaled) in models_reg.items():
    X      = X_test_scaled if scaled else X_test
    y_pred = model.predict(X)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    r2     = r2_score(y_test, y_pred)
    print(f"{name:<28} {rmse:>8.2f} {r2:>8.4f}")
    if rmse < best_rmse:
        best_rmse = rmse
        best_name = name

print("=" * 55)
print(f"Best Model: {best_name} (RMSE={best_rmse:.2f})")
print("=" * 55)

In [0]:
import time
from mlflow.tracking import MlflowClient

model_uri = f"runs:/{xgb_run_id}/xgboost_regressor_model"

rul_model_details = mlflow.register_model(
    model_uri=model_uri,
    name="PredMaint-RUL"
)

time.sleep(5)

client = MlflowClient()
client.set_registered_model_alias(
    name="workspace.default.predmaint-rul",
    alias="champion",
    version=rul_model_details.version
)

print(f"PredMaint-RUL registered")
print(f"Version: {rul_model_details.version}")
print(f"Alias:   champion")

In [0]:
print("=" * 55)
print("NOTEBOOK 08 COMPLETE — RUL REGRESSION")
print("=" * 55)
print(f"Experiment:       PredMaint-RUL")
print(f"Runs logged:      4 (LR + RF + XGB + Quantile)")
print(f"Best RMSE:        {best_rmse:.2f} cycles")
print(f"Uncertainty QR:   80% confidence intervals")
print(f"Model Registry:   PredMaint-RUL -> champion")
print("=" * 55)
print("\nYOUR CONTEST NUMBERS")
print(f"Classification F1:  0.9420")
print(f"Classification AUC: 0.9984")
print(f"RUL RMSE:           {best_rmse:.2f} cycles")
print("=" * 55)
print("READY FOR DAY 6 — Gold Layer + Dashboard")
print("=" * 55)